## The Xtrack Environment
For more info see [Environment Section in the User's guide](https://xsuite.readthedocs.io/en/latest/environment.html)

An xtrack **Environment** is the shared container that keeps together:

 - **Variables** in env.vars: scalar knobs and deferred expressions used to drive elements and lines.
 - **Elements** in env.elements: magnets, markers, RF cavities, etc. They can be reused across multiple lines.
 - **Lines** in env.lines: ordered sequences of elements assembled from the environment.
 - **Additional data** including reference particles and user-defined functions.

In [4]:
import xtrack as xt

### Create an environment

In [5]:
env = xt.Environment()

In [6]:
env

Environment(0 lines: {}, 0 elements, 1 vars, 0 particles)

### Define variables

In [7]:
env['l_q'] = 1.0
env['kq'] = 0.12
env['kq.trim'] = '0.05 * kq' # deferred expression (is updated when kq changes)
env['kq.total'] = 'kq + kq.trim' # deferred expression

In [8]:
# Access variables
env['kq'], env['kq.trim'], env['kq.total']

(0.12, 0.006, 0.126)

In [9]:
# Behavior of deferred expressions
env['kq'] = 0.06
env['kq'], env['kq.trim'], env['kq.total']

(0.06, 0.003, 0.063)

### Define elements (the tedious way)

In [10]:
# Elements can also be instantiated using their constructor and then added to env
myq = xt.Quadrupole(length=0, k1=0)
env.elements['myq'] = myq

# Deferred expressions can be used to control the element
env['myq'].length = 'l_q'
env['myq'].k1 = 'kq.total'

### Define elements (the compact way)

In [11]:
# Create elements with env.new
env.new('qf', xt.Quadrupole, length='l_q', k1='kq.total')
env.new('qd', xt.Quadrupole, length='l_q', k1='-kq.total')
env.new('dr', xt.Drift, length=6.0);

In [12]:
# Deferred expressions act on the elements
env['qf'].k1

np.float64(0.063)

In [13]:
env['kq'] = 0.12
env['qf'].k1

np.float64(0.126)

### Define beam lines

In [14]:
# Define a line using the elements above
line_fodo = env.new_line(components=['qf', 'dr', 'qd', 'dr'])

In [15]:
# The environment can always be retrieved from the line
line_fodo.env

Environment(0 lines: {}, 4 elements, 5 vars, 0 particles)

In [16]:
# A line can be stored in the environment
env['fodo'] = line_fodo
env

Environment(1 lines: {fodo}, 4 elements, 5 vars, 0 particles)

In [17]:
# When a name is passed to `env.new_line(...)`, the line is automatically stored in env
env.new_line(name='half_fodo', components=['qf', 'dr'])
env

Environment(2 lines: {fodo, half_fodo}, 4 elements, 5 vars, 0 particles)

### Inspect environment containers

In [18]:
env

Environment(2 lines: {fodo, half_fodo}, 4 elements, 5 vars, 0 particles)

In [19]:
# Inspect variables
env.vars

EnvVars(5 vars: {t_turn_s, l_q, kq, kq.trim, kq.total, ...})

In [20]:
env.vars.get_table()

VarsTable: 5 rows, 3 cols
name             value expr          
t_turn_s             0 None          
l_q                  1 None          
kq                0.12 None          
kq.trim          0.006 (0.05 * kq)   
kq.total         0.126 (kq + kq.trim)

In [21]:
# Inspect elements
env.elements.get_table()

LineTable: 4 rows, 9 cols
name element_type isthick isreplica parent_name parent_type prototype iscollective        length
dr   Drift           True     False None        None        None             False             6
myq  Quadrupole      True     False None        None        None             False             1
qd   Quadrupole      True     False None        None        None             False             1
qf   Quadrupole      True     False None        None        None             False             1

In [22]:
# Inspect lines
env.lines.get_table()

Table: 2 rows, 3 cols
name      num_elements mode  
fodo                 4 normal
half_fodo            2 normal